# HASS training · remote control center
Configure the visible `training.env` next to this notebook once. Select a model, frozen dataset and run label below. Every job executes remotely; Jupyter displays receipts, metrics and a bounded live log. Local records are saved in `runs/<label>/<stage>/`. No adapter weights or dataset shards are downloaded here.
Use `hass-dataset.ipynb` to make a new dataset. Reuse its release receipt for all three models. For the existing job 8 adapter, set SOURCE_RUN_ID and skip training after preflight. The dataset must match that source run.

In [ ]:
from pathlib import Path
from copy import deepcopy
import json, re, sys
import yaml
WORKSPACE = Path.home() / 'eliza/training'
RUN_LABEL = 'hass-qwen08-01'
MODEL_PRESET = 'sft_hass_private_smoke.template.yaml'
MANIFEST_URI = ''  # or leave blank and select DATASET_LABEL below
MANIFEST_SHA256 = ''
DATASET_LABEL = 'hass-household-01'
SOURCE_RUN_ID = ''  # archived SFT UUID; empty trains a new adapter
TRAINING_OVERRIDES = {}  # e.g. {'max_steps': 100}
RESOURCE_OVERRIDES = {}
SETTING_OVERRIDES = {}
if (WORKSPACE / '.client/nlp_training').is_dir():
    sys.path.insert(0, str(WORKSPACE / '.client'))
from nlp_training.operator import connect_workspace, submit_prepared, read_report, watch_run, format_status
from IPython.display import clear_output
settings, storage, training, dagster = connect_workspace(WORKSPACE, SETTING_OVERRIDES)
REPO = Path(settings['NLP_LAB_REPO']).expanduser()
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', RUN_LABEL):
    raise ValueError('Choose an alphanumeric run label')
RUN = WORKSPACE / 'runs' / RUN_LABEL
RUN.mkdir(parents=True, exist_ok=True)
if not MANIFEST_URI:
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', DATASET_LABEL):
        raise ValueError('Invalid dataset label')
    release = json.loads((WORKSPACE / 'runs' / DATASET_LABEL / 'release.json').read_text())
    MANIFEST_URI, MANIFEST_SHA256 = release['manifest_uri'], release['manifest_sha256']
pin = dict(uri=MANIFEST_URI, sha256=MANIFEST_SHA256)
match = re.fullmatch(r's3://([^/]+)/dataset_id=([^/]+)/dataset_version=([^/]+)/manifest\.json', MANIFEST_URI)
if not match or not re.fullmatch(r'[a-f0-9]{64}', MANIFEST_SHA256):
    raise ValueError('Enter the frozen manifest URI and checksum')
bucket, dataset_id, version = match.groups()
cfg = yaml.safe_load((REPO / 'configs' / MODEL_PRESET).read_text())
cfg['data'].update(bucket=bucket, dataset_id=dataset_id, dataset_version=version, expected_manifest_sha256=MANIFEST_SHA256)
cfg['tracking']['mlflow_tracking_uri'] = settings['MLFLOW_TRACKING_URI']
cfg['training'].update(TRAINING_OVERRIDES)
resources = {'cpus': 8, 'memory_mb': 32768, 'time_minutes': 120, **RESOURCE_OVERRIDES}
(RUN / 'plan.yaml').write_text(yaml.safe_dump(dict(model_preset=MODEL_PRESET, dataset=pin, config=cfg, resources=resources)))
print('Run directory:', RUN)
print('MLflow server:', cfg['tracking']['mlflow_tracking_uri'])
print('Model:', cfg['model'], '| Resources:', resources)
def launch(kind, config):
    return submit_prepared(training, REPO, RUN / kind, kind=kind, config=config, pin=pin, resources=resources, settings=settings)
def watch(kind, run_id):
    def render(state, tail):
        clear_output(wait=True)
        print(format_status(state))
        print('Local run records:', RUN / kind)
        print('--- recent remote output ---')
        print(tail[-6000:] or 'Waiting for runtime/bootstrap log output…')
    return watch_run(training, run_id, RUN / kind, render=render)
def report(kind, run_id, name):
    return read_report(training, run_id, name, run_dir=RUN / kind)


## Dataset preview
Read the frozen manifest and split counts. This is metadata only; sample conversations and split-group coverage are displayed by the dataset notebook.

In [ ]:
import hashlib
manifest_bytes = storage.read_bytes(bucket=bucket, key=MANIFEST_URI.split('/', 3)[3], max_bytes=32*1024*1024)
if hashlib.sha256(manifest_bytes).hexdigest() != pin['sha256']:
    raise ValueError('Manifest checksum mismatch')
manifest = json.loads(manifest_bytes)
print('Frozen stats:', manifest['stats'])
print('Split files:', len(manifest['splits']))


## Remote preflight and MLflow verification
This submits a CPU-only Slurm job on DGX. It creates a labelled connectivity run on the configured MLflow server, writes and reads back a probe metric, then measures every dataset row with the pinned tokenizer. Failure stops here, before training. Nothing runs locally besides API requests.

In [ ]:
context_id = launch('hass-context', cfg)
watch('hass-context', context_id)
context = report('hass-context', context_id, 'context-preflight.json')
print('Verified MLflow:', context['mlflow']['url'])
context


## Training
Skip if evaluating a prior source run. Otherwise review the plan and submit. Run labels bind immutable plans: choose a new label for a changed model/config. A reconnect reuses the journaled UUID; it never resubmits automatically. The monitor shows stages, step/total and recent remote logs. A terminal worker receipt plus verified report is required for success.

In [ ]:
if SOURCE_RUN_ID:
    print('Using archived source run:', SOURCE_RUN_ID)
else:
    if not context['fits_native_context'] or context['manifest_sha256'] != pin['sha256'] or not context['mlflow']['verified']:
        raise ValueError('Preflight did not qualify this dataset and MLflow destination')
    train_cfg = deepcopy(cfg)
    train_cfg['model']['max_seq_len'] = context['suggested_budget']
    trained_id = launch('sft', train_cfg)
    watch('sft', trained_id)
    summary = report('sft', trained_id, 'training-summary.json')
    print('Steps:', summary['optimizer_steps'], '/', summary['requested_steps'])
    print('MLflow:', summary['mlflow_tracking_uri'].rstrip('/') + '/#/experiments/' + summary['mlflow_experiment_id'] + '/runs/' + summary['mlflow_run_id'])
    print('Verified report:', RUN / 'sft/training-summary.json')


## Remote adapter reload and baseline comparison
DGX verifies the archived adapter, reloads it and compares against the original checkpoint on fixed held-out cases. Jupyter shows the resulting private report. This does not deploy the adapter to Eliza Small or execute Home Assistant calls.

In [ ]:
eval_cfg = deepcopy(cfg if SOURCE_RUN_ID else train_cfg)
eval_cfg['evaluation'] = {'source_run_id': SOURCE_RUN_ID or trained_id}
eval_id = launch('hass-eval', eval_cfg)
watch('hass-eval', eval_id)
comparison = report('hass-eval', eval_id, 'held-out-comparison.json')
comparison


## Recovery and explicit restoration
For a lost submission response, use the UUID in `runs/<label>/<stage>/run.json` with `training.recover_submission(uuid)` after `training.discover()`. Logs, reports and status remain in the same folder after restart. Never delete a submission marker to retry an uncertain job. Restore inference only when all jobs have finished.

In [ ]:
if input('Type RESTORE to return the DGX to inference: ').strip() == 'RESTORE':
    print(training.environment('inference').get('phase'))
